# k-Means From Scratch

**Topic:** Clustering &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X` `(1800, 6)`: points from **5 clusters** of different sizes and spreads (some elongated). No labels are given to you.
- The true cluster assignments are created in the data cell below (as `_y_true_clusters`) and read by the evaluation cell, which scores your labels with the Adjusted Rand Index (so your cluster *numbering* doesn't matter).

## Your task

With **NumPy only** (no `sklearn.cluster`):

1. Implement Lloyd's algorithm for k = 5. Run it from ~10 random initializations and keep the run with the lowest inertia (sum of squared distances to assigned centroids).
2. Set `predictions` to your `(1800,)` array of cluster labels for `X`.

## Evaluation

Adjusted Rand Index vs the hidden truth must be ≥ 0.75.


In [ ]:
import numpy as np

rng = np.random.default_rng(301)
k, d = 5, 6
centers = rng.normal(0, 5.0, (k, d))
sizes = np.array([500, 420, 350, 300, 230])
stretch = [np.diag(rng.uniform(0.5, 1.8, d)) for _ in range(k)]

_parts, _labels = [], []
for c in range(k):
    pts = rng.normal(size=(sizes[c], d)) @ stretch[c] + centers[c]
    _parts.append(pts); _labels.append(np.full(sizes[c], c))
X = np.vstack(_parts)
_y_true_clusters = np.concatenate(_labels)
_perm = rng.permutation(len(X))
X, _y_true_clusters = X[_perm], _y_true_clusters[_perm]

print(f"X {X.shape}: find the 5 groups (no labels for you!)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: Adjusted Rand Index against the hidden true clusters
# =====================================================================
import numpy as np
from sklearn.metrics import adjusted_rand_score

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
if _pred.shape != _y_true_clusters.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_y_true_clusters.shape}.")

_ari = float(adjusted_rand_score(_y_true_clusters, _pred))
print("=" * 52)
print(f"Adjusted Rand Index : {_ari:.4f}   (target: >= 0.75)")
print(f"Your cluster sizes  : {np.bincount(_pred.astype(int)).tolist()}")
print("=" * 52)
print("PASSED: clean recovery!" if _ari >= 0.75 else "NOT PASSED yet: more restarts, or check your update step.")


## Solution

Lloyd's algorithm: assign each point to its nearest centroid, recompute centroids as cluster means, repeat until stable. Restart from several random initializations and keep the run with the lowest inertia to dodge bad local optima.

```python
import numpy as np

def lloyd(X, k, seed, iters=100):
    r = np.random.default_rng(seed)
    centroids = X[r.choice(len(X), k, replace=False)]
    for _ in range(iters):
        d = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = d.argmin(axis=1)
        new = np.vstack([
            X[labels == c].mean(axis=0) if (labels == c).any() else centroids[c]
            for c in range(k)
        ])
        if np.allclose(new, centroids):
            break
        centroids = new
    inertia = float((np.linalg.norm(X - centroids[labels], axis=1) ** 2).sum())
    return labels, inertia

best_labels, best_inertia = None, np.inf
for seed in range(10):
    labels, inertia = lloyd(X, 5, seed)
    if inertia < best_inertia:
        best_labels, best_inertia = labels, inertia

print(f"best inertia: {best_inertia:.1f}")
predictions = best_labels
```
